# Earthquake News Dataset Analysis and Design Workflow

This notebook collects, processes, and analyses English-language earthquake-related news data. The workflow integrates data scraping, text vectorisation, machine learning, and visualisation, forming a pipeline for data-driven design in Blender.

In [14]:
!pip install requests pandas textblob scikit-learn matplotlib seaborn


[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: C:\Users\24586\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


In [15]:
import requests
import pandas as pd
import time
import os
from textblob import TextBlob
import matplotlib.pyplot as plt
import seaborn as sns

In [16]:
API_KEY = "ec08f85b62dc49708602f72af02a5966"

In [17]:
os.makedirs("output/data", exist_ok=True)
os.makedirs("output/images", exist_ok=True)

In [18]:
queries = [
    "earthquake",
    "earthquake disaster",
    "seismic damage",
    "earthquake rescue"
]

all_articles = []

for q in queries:
    print(f"Collecting: {q}")
    
    for page in range(1, 4):
        params = {
            "q": q,
            "language": "en",
            "pageSize": 50,
            "page": page,
            "apiKey": API_KEY
        }

        response = requests.get("https://newsapi.org/v2/everything", params=params)
        data = response.json()

        for a in data.get("articles", []):
            all_articles.append({
                "title": a["title"],
                "description": a["description"],
                "source": a["source"]["name"],
                "date": a["publishedAt"]
            })
        
        time.sleep(1)

df = pd.DataFrame(all_articles)
print("Raw data:", len(df))

Collecting: earthquake
Collecting: earthquake disaster
Collecting: seismic damage
Collecting: earthquake rescue
Raw data: 276


In [19]:
df = df.dropna(subset=["title"])
df = df.drop_duplicates(subset=["title"])

df["description"] = df["description"].fillna("")
df["text"] = df["title"] + " " + df["description"]

print("Cleaned data:", len(df))
df.head()

Cleaned data: 232


,title,description,source,date,text
0,Japan Under Tsunami Warning Following M7.7 Qua...,Japanese officials issued a tsunami advisory a...,Gizmodo.com,2026-04-20T14:05:03Z,Japan Under Tsunami Warning Following M7.7 Qua...
1,'My sister loved to party and turned her life ...,Thousands attend a retreat in Derry named afte...,BBC News,2026-04-16T09:10:32Z,'My sister loved to party and turned her life ...
2,Tsunami Alert Issued After 7.5-Magnitude Earth...,A magnitude 7.5 earthquake struck of the north...,Yahoo Entertainment,2026-04-20T11:24:56Z,Tsunami Alert Issued After 7.5-Magnitude Earth...
3,Magnitude 5.7 earthquake reported in Nevada,A magnitude 5.5 earthquake was reported near S...,Abcnews.com,2026-04-14T07:00:05Z,Magnitude 5.7 earthquake reported in Nevada A ...
4,4.9 magnitude earthquake rattles Northern Cali...,A 4.9 magnitude earthquake shook Northern Cali...,Abcnews.com,2026-04-02T09:31:52Z,4.9 magnitude earthquake rattles Northern Cali...


In [20]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(stop_words="english")
X_tfidf = tfidf.fit_transform(df["text"])

print("TF-IDF done:", X_tfidf.shape)

TF-IDF done: (232, 2590)


In [21]:
from sklearn.feature_extraction.text import CountVectorizer

count = CountVectorizer(stop_words="english")
X_count = count.fit_transform(df["text"])

print("Count done:", X_count.shape)

Count done: (232, 2590)


In [22]:
def classify_sentiment(score):
    if score > 0.1:
        return "positive"
    elif score < -0.1:
        return "negative"
    else:
        return "neutral"

df["sentiment_score"] = df["text"].apply(lambda x: TextBlob(x).sentiment.polarity)
df["sentiment_label"] = df["sentiment_score"].apply(classify_sentiment)

In [23]:
keywords = ["death", "damage", "rescue", "collapse", "injured", "homeless"]

for k in keywords:
    df[k] = df["text"].str.contains(k, case=False).astype(int)

keyword_counts = df[keywords].sum()
print(keyword_counts)

death        3
damage      13
rescue       2
collapse     1
injured      2
homeless     0
dtype: int64


In [24]:
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=4, random_state=42)
df["cluster"] = kmeans.fit_predict(X_tfidf)

In [25]:
# Sentiment
plt.figure()
sns.countplot(x=df["sentiment_label"])
plt.title("Sentiment Categories")
plt.savefig("output/images/sentiment.png", dpi=300, bbox_inches='tight')
plt.close()

# Keywords
plt.figure()
keyword_counts.plot(kind="bar")
plt.title("Keyword Frequency")
plt.savefig("output/images/keywords.png", dpi=300, bbox_inches='tight')
plt.close()

# Cluster
plt.figure()
sns.countplot(x=df["cluster"])
plt.title("Cluster Distribution")
plt.savefig("output/images/cluster.png", dpi=300, bbox_inches='tight')
plt.close()

print("All plots saved!")

All plots saved!


In [26]:
from sklearn.decomposition import PCA

X_dense = X_tfidf.toarray()

pca = PCA(n_components=2)
X_reduced = pca.fit_transform(X_dense)

plt.figure()
plt.scatter(X_reduced[:,0], X_reduced[:,1], c=df["cluster"])
plt.title("PCA Clusters")
plt.savefig("output/images/pca.png", dpi=300, bbox_inches='tight')
plt.close()

In [27]:
# 完整数据
df.to_csv("output/data/full_dataset.csv", index=False)

# Blender数据
df_export = df[[
    "sentiment_score",
    "cluster",
    "death",
    "damage",
    "rescue",
    "collapse"
]]

df_export.to_csv("output/data/blender_input.csv", index=False)

print("All data exported!")

All data exported!


TF-IDF and Count Vectorizer were used to compare text representation methods. TF-IDF highlights important terms by reducing the weight of common words, while Count Vectorizer captures raw word frequency.